In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.homology import VietorisRipsPersistence
from gtda.time_series import TakensEmbedding, SlidingWindow
import plotly.express as px
from sklearn.metrics import mutual_info_score
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from gtda.plotting import plot_heatmap

In [ ]:
slug_10df = pd.read_csv('processed_EA-10.csv')
slug_10df

From the dataframe for Well EA-10 I am carving out the top hole pressure data, to analyze by means of TDA and Persistent Homology. \
The idea is that by embedding the time seires signal in a high dimensional space (Takens Embedding) this form a point cloud. As a periodic oscillation appears in the time series, the point cloud should turn into an orbit in a phase space, and persistent homology should be the right tool to idenity loops and orbits in such space. 

In [ ]:
slug_july = slug_10df[(slug_10df['TimeStamp'] > "2022-06-27 01:00:00") & (slug_10df['TimeStamp'] < "2022-07-04 12:00:00")]


from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=5, cols=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['TH_Pressure'],  name="TH_Pressure"), row=1, col=1)
fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['FlowLine_Pressure'],  name="FlowLine_Pressure"), row=1, col=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['GL_Flowrate'],  name="GL_Flowrate"), row=2, col=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['GL_Pressure'],  name="GL_Pressure"), row=3, col=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['BH_Pressure'],  name="BH_Pressure"), row=4, col=1)
#fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['WCV_Status'],  name="WCV_Status"), row=5, col=1)
fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['BH_Temperature'],  name="BH_Temperature"), row=5, col=1)


fig.update_layout(height=600, width=1200, title_text="Signal Analysis")
fig.show()


In [ ]:
slug_10ts = slug_10df[['TimeStamp', 'TH_Pressure']]
slug_10ts['TimeStamp'] = pd.to_datetime(slug_10ts['TimeStamp'])
slug_10ts.set_index('TimeStamp', inplace=True)
slug_10ts

In [ ]:
figure_FHP = px.scatter()
figure_FHP.add_scatter(x=slug_10ts.index, y=slug_10ts['TH_Pressure'],  name="FHP_well10")
figure_FHP.show()

Of the pressure signal in well10 (top hole pressure) I am extracting a period in which there is some variance (slugging - no slugging) to test both TDA and FFT based analytics. The beginning of July 2022 looks like the ideal place to start. 

In [ ]:
slug_july = slug_10ts["2022-06-27 01:00:00":"2022-07-4 12:00:00"]

figure_FHP = px.scatter()
figure_FHP.add_scatter(x=slug_july.index, y=slug_july['TH_Pressure'],  name="FHP_well10")
figure_FHP.show()

In [ ]:
slug_july

From previous tests, it looks like TDA and PH are perform poorly when the time resolution is low. Here with a peak frequency of about 6 minutes, anf a 1 minute sampling rate this is definitely too choppy. I therefore upsample by a cubic spline interpolation to a 6 s sampling, to have about 60 points per oscillation period. 

In [ ]:
from scipy.interpolate import CubicSpline

xmin = 1
xmax = len(slug_july['TH_Pressure'])-1
some_step = 0.1                         # this is a 6 seconds step, 10 points per minute
cs = CubicSpline(range(len(slug_july)), slug_july['TH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)

slug_july.insert(1, "idx", np.linspace(0, len(slug_july)-1, num=len(slug_july)), True)

print(cs(x_range).shape)

# upsampled to 10 times


In [ ]:
new_range = pd.date_range(slug_july.index[0], slug_july.index.values[-1], freq='6S')
new_range = new_range[0:len(cs(x_range))]

slug_july_spline = []
slug_july_spline = pd.DataFrame(data=cs(x_range)) 
slug_july_spline.set_index(new_range, inplace=True)
slug_july_spline.rename(columns={"0":'TH_Pressure'}) # not sure why this is not working
slug_july_spline.index.name='TimeStamp'

slug_july_spline

In [ ]:
figure_FHP = px.scatter()
figure_FHP.add_scatter(x=slug_july_spline[:1000].index, y=slug_july_spline[:1000][0], name="FHP_well10")
figure_FHP.show()

We need to find the sweet spot for the analysis, a time window that makes sense and that is not too large. 
From my tests, it looks like 30 minutes is a good trade off (300 points)

In [ ]:

fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))

freq_units = 0.1/len(slug_july_spline[700:1000])
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('EA-36 THP stats')
data[0].set_xlabel('time (10s)')
data[0].set_ylabel('GL Flowrate')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[2].set_ylabel('power spectrum')

data[1].set_yscale('log')
data[1].set_xscale('log')
data[2].set_xlim([1,60])
data[2].set_ylim([-1,5.5E3])

ss_THP = slug_july_spline[0][700:1000]
 
fft_ss = np.fft.rfft(ss_THP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)

data[0].plot(ss_THP)
data[1].plot(power_spectrum_ss)
data[2].plot(power_spectrum_ss)

In [ ]:
max_time_delay =15
max_embedding_dimension = 6
stride = 1

signal = slug_july_spline[0][700:1000]

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")


embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(signal)
pca = PCA(n_components=3)

y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)   
plot_point_cloud(y_ss_BHP_pca)


This circular pattern is the signature of periodicity. In homology, this should be a $H_1$ generator, i.e. produce a "high persistence" point in the homology dimension 1 (meaning that the loop 'hole" is a persistent feature of the shape of data).\
This can become evident by analyzing the point cloud with persistence diagrams. High persistence points appear away from the diagonal of the graph.

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram = VRP.fit_transform_plot(y_ss_BHP[None,:,:])

This is definitely some good signal with of homology dimension 1, therfore the approach is detecting periodic components in the pressure signal. \

Now splitting the signal into windows of 30 minutes each, each one starting 20 minutes from each starting point. 

In [ ]:
from gtda.time_series import SlidingWindow, TakensEmbedding

signal = slug_july_spline[0]
print(len(signal))

window_size = 300 
window_stride = 200
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(signal)

print(X_windows.shape)


Now I determine the optimal embedding parameters for each time window and find out which ones to use for the whole duration of the time series.

In [ ]:
def batch_analyzer(input_df, stride, max_embedding_dimension, max_time_delay):
    max_time_delay = int(max_time_delay)
    max_embedding_dimension = int(max_embedding_dimension)

    i = 0
    averages = np.zeros(shape=(len(input_df),2))
    
   # print('max time delay:',max_time_delay, 'max dim:',max_embedding_dimension)
    
    for timeserie in input_df:
        
        optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
            timeserie, max_time_delay, max_embedding_dimension, stride=stride
            )
        if optimal_embedding_dimension < 3:
            optimal_embedding_dimension = 3
            
        # print('analyzing nr.',i, 'dim',optimal_embedding_dimension,'delay', optimal_time_delay)
        averages[i] = [optimal_embedding_dimension, optimal_time_delay]
        i += 1

    print('Max Dimension:', averages.max(0))
    print('Mean Dimension:', averages.mean(0))
    return averages

In [ ]:
stats = batch_analyzer(X_windows, 1, 13, 21)


In [ ]:
for i in range(len(stats)):
    if stats[i][0] == 13:
        print(i, stats[i])
    elif stats[i][1] == 21:
        print(i, stats[i])


By splitting the signal in time windows of 30 minutes (300 points) each starting 20 minutes after the beginning of the preceding one (so to have 10 min overlap) it looks like the optimal embedding parameters are somewhat close to $\tau$=15 and dim=8 (better to be larger than underestimating).

In [ ]:
signal = X_windows[195]

embedding_dimension = 8
embedding_time_delay = 15
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(signal)
pca = PCA(n_components=3)

y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)   
plot_point_cloud(y_ss_BHP_pca)

In [ ]:
optimal_time_delay = 13
optimal_embedding_dimension = 9
stride = 1

In [ ]:
from gtda.pipeline import Pipeline
from gtda.metaestimators import CollectionTransformer
from gtda.diagrams import PersistenceEntropy

embedder = TakensEmbedding(time_delay=optimal_time_delay,
                           dimension=optimal_embedding_dimension,
                           stride=stride)
batch_pca =  CollectionTransformer(PCA(n_components=3),n_jobs=-1)
persistence = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=None)
entropy = PersistenceEntropy(nan_fill_value=-10)
steps = [
         ("embedder", embedder),
         ("pca", batch_pca),
         ("persistence", persistence),
         #("entropy", entropy)
        ]
topological_transfomer = Pipeline(steps)

In [ ]:
signal_persistence = topological_transfomer.fit_transform(X_windows)

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in signal_persistence: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
fig = px.line(title='Maximum Persistence, indexed by sliding window number, window of 30 min')
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.show() 

In [ ]:
embedder = TakensEmbedding(time_delay=optimal_time_delay,
                           dimension=optimal_embedding_dimension,
                           stride=stride)
batch_pca =  CollectionTransformer(PCA(n_components=3),n_jobs=-1)
persistence = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=None)
entropy = PersistenceEntropy(nan_fill_value=-10)
steps = [
         ("embedder", embedder),
         ("pca", batch_pca),
         ("persistence", persistence),
         ("entropy", entropy)
        ]
topological_transfomer = Pipeline(steps)

In [ ]:
signal_entropy = topological_transfomer.fit_transform(X_windows)

In [ ]:
Ent_H0 = []
Ent_H1 = []

for entropy in signal_entropy: 
    Ent_H0.append(entropy[0])
    Ent_H1.append(entropy[1])

Entropy0 = pd.DataFrame(Ent_H0)    
Entropy0 = Entropy0[0].rolling(25).mean()
Entropy0.dropna(inplace=True)
Entropy1 = pd.DataFrame(Ent_H1)    
Entropy1 = Entropy1[0].rolling(25).mean()
Entropy1.dropna(inplace=True)

fig = px.line(title='Persistent Entropy')
fig.add_scatter(y=Entropy0, name="Entropy in homology dimension H_0")
fig.add_scatter(y=Entropy1, name="Entropy in homology dimension H_1")
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")

fig.show() 

While it looks there is a decent correlation between slugging and the maximum persistence in $H_1$, this is much less clear for the persistence entropy, which looks all over the place. I therefore stick to the persistence for the time being. \
I am also having a look at the Betti numbers, which were useful in similar situations before. 

In [ ]:
TE = TakensEmbedding(time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=1)
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PE = PersistenceEntropy()
PE_norm = PersistenceEntropy(normalize=True)
Betti = BettiCurve()

slugging_signals = X_windows
slugging_point_cloud  = TE.fit_transform(slugging_signals)
slugging_diagrams = VRP.fit_transform(slugging_point_cloud)
slugging_entropy = PE.fit_transform(slugging_diagrams)
slugging_entropynorm = PE_norm.fit_transform(slugging_diagrams)
slugging_Betti = Betti.fit_transform(slugging_diagrams)

In [ ]:
import statistics as stats

meanBettiH1 = []
medianBettiH1 = []
modeBettiH1 = []

for i in range(len(slugging_Betti[:,1,0])):
    meanBettiH1.append(stats.mean(slugging_Betti[i,1,:]))
    medianBettiH1.append(stats.median(slugging_Betti[i,1,:]))
    modeBettiH1.append(stats.mode(slugging_Betti[i,1,:]))
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH1), name="Mean of Betti curve H_1")
#figBetti.add_scatter(y=(medianBettiH1), name="Median of Betti curve H_1")
#figBetti.add_scatter(y=(modeBettiH1),name="Mode of Betti curve H_1")
figBetti.show()

In [ ]:
slug_july_spline['PersH0'] = np.NaN
slug_july_spline['PersH1'] = np.NaN
slug_july_spline['Entropy0'] = np.NaN
slug_july_spline['Entropy1'] = np.NaN
slug_july_spline['signal_max'] = np.NaN
slug_july_spline['signal_min'] = np.NaN
slug_july_spline['signal_avg'] = np.NaN
slug_july_spline['signal_std'] = np.NaN


# Setting thresholds for slugging identification (arbitrary)

limit_PersH0 = 0.5
limit_PersH1 = 1.0

i = 0 
for time in slug_july_spline.index[::200]:
    if i == len(X_windows):
        break
    if i != 0: 
        slug_july_spline['signal_max'][time] = max(X_windows[i])
        slug_july_spline['signal_min'][time] = min(X_windows[i])
        slug_july_spline['signal_avg'][time] = stats.mean(X_windows[i])
        slug_july_spline['signal_std'][time] = stats.stdev(X_windows[i])
        # setting high value just for visibility and graphing
        if max_pers_H0[i] > limit_PersH0: 
            slug_july_spline['PersH0'][time] = 20.0
        if max_pers_H1[i] > limit_PersH1: 
            slug_july_spline['PersH1'][time] = 20.0

    i += 1

slug_july_spline.fillna(0.0, inplace=True)
figBetti = px.line()
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['PersH1'],  name="Pers H1")
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['PersH0'],  name="Pers H0")
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline[0],  name="FHP_well10")


figBetti.show()


Okay, but still the maximum persistence in $H_1$ does the job best.  

In [ ]:
slug_july_spline.fillna(0.0, inplace=True)
figBetti = px.line()
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['PersH1'],  name="Pers H1")
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['signal_std'],  name="Standard Dev of TH Pressure")
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline[0],  name="FHP_well10")

Same analysis but doing FFT of each time window. 

In [ ]:
slug_july_spline

In [ ]:
from scipy.signal import periodogram

fs = 1/6            # sampling every 6 seconds
dominant_freq = []

for signal in X_windows:
    f, Pxx_den = periodogram(signal, fs)
    dominant_freq.append(f[np.argmax(Pxx_den)])

figDominant = px.line()
figDominant.add_scatter(y=dominant_freq)


In [ ]:
slug_july_spline['Freq(Hz)'] = np.NaN

i = 0 
for time in slug_july_spline.index[::200]:
    if i == len(X_windows):
        break
    if i != 0:
        #print(dominant_freq[i])
        slug_july_spline['Freq(Hz)'][time] = dominant_freq[i]
    i += 1

In [ ]:
slug_july_spline.fillna(0.0, inplace=True)

figBetti = px.line()
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['Freq(Hz)']*1000,  name="Dominant Freq")
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline[0],  name="FHP_well10")

Well, the frequency of the leading FFT component is not correlating too well with the slugging signal. See for example the results at July 2nd and July 4th, where there is high signal even without any evident slugging. \
Probably it is not a good indicator for $identification$ of slugs but may be useful elsewhere. This is also to be very dependent on the window size as I am cutting away the lower frequency components.  

In [ ]:
slug_july_spline

In [ ]:
fs = 1/6

for signal in X_windows:
    f, Pxx_den = periodogram(signal, fs)
    dominant_freq.append(f[np.argmax(Pxx_den)])
    exit

figDominant = px.line()
figDominant.add_scatter(x=f, y=Pxx_den, name='periodogram TH_pressure')


A dominant peak at fequency 0.00277 Hz coresponds to a 6 minutes periodicity, which makes sense. 